#CSV Ingestion (Chunk 1) via COPY INTO

In [0]:
# Notebook: 03_bronze_copy_into_FIXED
CATALOG = "vstone_catalog"
BRONZE = "bronze"
TABLE_NAME = f"{CATALOG}.{BRONZE}.listings_csv_copyinto"
FILE_PATH = f"/Volumes/{CATALOG}/raw/chunks/1_main_chunk_1.csv"

# 1. DROP old table to fix the schema mismatch
spark.sql(f"DROP TABLE IF EXISTS {TABLE_NAME}")

# 2. CREATE TABLE with explicit STRING columns (matching your CSV headers)
# Isse trailing zeros (.0) ka lafda khatam ho jayega
spark.sql(f"""
CREATE TABLE {TABLE_NAME} (
    cost STRING, currency STRING, marka STRING, model STRING, year STRING,
    has_license STRING, place STRING, date STRING, id STRING, engine STRING,
    power STRING, gear STRING, probeg STRING, sWheel STRING, complectation STRING,
    transmission STRING, R STRING, G STRING, B STRING,
    load_dt TIMESTAMP, 
    source_file STRING
) USING DELTA
""")

# 3. COPY INTO without inferSchema
# 'inferSchema' = 'false' ensures data is treated exactly as text
spark.sql(f"""
COPY INTO {TABLE_NAME}
FROM '{FILE_PATH}'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false') 
COPY_OPTIONS ('mergeSchema' = 'true')
""")

# 4. Audit Columns Update
spark.sql(f"""
UPDATE {TABLE_NAME} 
SET load_dt = current_timestamp(), source_file = '1_main_chunk_1.csv' 
WHERE load_dt IS NULL
""")

print(f"✅ Table {TABLE_NAME} re-created and loaded with strict String schema.")

In [0]:
%sql
select * from vstone_catalog.bronze.listings_csv_copyinto limit 5;